In [3]:
!pip install ucimlrepo
!pip install scikeras
!pip install --upgrade scikit-learn scikeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 62.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [4]:
from ucimlrepo import fetch_ucirepo


car_evaluation = fetch_ucirepo(id=19)


X = car_evaluation.data.features
y = car_evaluation.data.targets


print(car_evaluation.metadata)

print(car_evaluation.variables)


{'uci_id': 19, 'name': 'Car Evaluation', 'repository_url': 'https://archive.ics.uci.edu/dataset/19/car+evaluation', 'data_url': 'https://archive.ics.uci.edu/static/public/19/data.csv', 'abstract': 'Derived from simple hierarchical decision model, this database may be useful for testing constructive induction and structure discovery methods.', 'area': 'Other', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1728, 'num_features': 6, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5JP48', 'creators': ['Marko Bohanec'], 'intro_paper': {'ID': 249, 'type': 'NATIVE', 'title': 'Knowledge acquisition and explanation for multi-attribute decision making', 'authors': 'M. Bohanec, V. Rajkovič', 'venue': '8th Intl Workshop on Expert Systems and their Applications, 

In [6]:
from ucimlrepo import fetch_ucirepo

car_evaluation = fetch_ucirepo(id=19)

print(type(car_evaluation))
print(type(X))

<class 'ucimlrepo.dotdict.dotdict'>
<class 'pandas.core.frame.DataFrame'>


In [ ]:
X.buying.replace(('vhigh','high','med','low'),(1,2,3,4), inplace=True)
X.maint.replace(('vhigh','high','med','low'),(1,2,3,4), inplace=True)
X.doors.replace(('2','3','4','5more'),(1,2,3,4), inplace=True)
X.persons.replace(('2','4','more'),(1,2,3), inplace=True)
X.lug_boot.replace(('small','med','big'),(1,2,3), inplace=True)
X.safety.replace(('low','med','high'),(1,2,3), inplace=True)
y['class'].replace(('unacc','acc','good','vgood'),(1,2,3,4), inplace=True)
y['class'] = y['class'] - 1

/tmp/ipython-input-29633739.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X.buying.replace(('vhigh','high','med','low'),(1,2,3,4), inplace=True)
/tmp/ipython-input-29633739.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X.buying.replace(('vhigh','high','med','low'),(1,2,3,4), inplace=True

In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import SGD

from scikeras.wrappers import KerasClassifier


#digits = load_digits()
#X, y = digits.data, digits.target


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


def create_model(num_layers=1, num_neurons=64, activation='relu', dropout_rate=0.0, momentum=0.9, learning_rate = 0.01):
    model = Sequential()

    model.add(Input(shape=(X_scaled.shape[1],)))

    for _ in range(num_layers):
        model.add(Dense(num_neurons, activation=activation))
        model.add(Dropout(dropout_rate))

    model.add(Dense(4, activation='softmax'))

    optimizer = SGD(learning_rate=learning_rate, momentum=momentum)


    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Define parameters grid
# Note: SciKeras expects the parameter names to match the create_model arguments
#
# param_grid = {
#     'model__num_layers': [2, 3],
#     'model__num_neurons': [32, 64],
#     'model__activation': ['relu', 'tanh'],
#     'model__dropout_rate': [0.2, 0.5],
#     'model__momentum': [0.5, 0.9]
# }
# param_grid = {
#     'model__num_layers': [2],
#     'model__num_neurons': [32],
#     'model__activation': ['relu'],
#     'model__dropout_rate': [0.2],
#     'model__momentum': [0.5]
# }
# param_grid = {

#     'model__num_layers': [1, 2, 3],


#     'model__num_neurons': [80],


#     'model__activation': ['relu', 'tanh'],


#     # 'model__dropout_rate': [0.1, 0.3, 0.5],
#     'model__dropout_rate': [0.1, 0.3],


#     'model__learning_rate': [0.001],


#     'batch_size': [16, 32]
# }
param_grid = {

    'model__num_layers': [1, 2, 3],


    'model__num_neurons': [64, 128, 256],


    'model__activation': ['relu', 'tanh'],


    'model__dropout_rate': [0.1, 0.3, 0.5],


    'model__learning_rate': [0.01, 0.001],


    'batch_size': [16, 32]
}

model = KerasClassifier(model=create_model, epochs=300, batch_size=32, verbose=0)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=kfold, scoring='accuracy',verbose=3)
grid_result = grid_search.fit(X_scaled, y)

print(f"Best: {grid_result.best_score_:.4f} using {grid_result.best_params_}")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
[CV 1/5] END batch_size=16, model__activation=relu, model__dropout_rate=0.1, model__learning_rate=0.01, model__num_layers=1, model__num_neurons=64;, score=0.994 total time=  41.7s
[CV 2/5] END batch_size=16, model__activation=relu, model__dropout_rate=0.1, model__learning_rate=0.01, model__num_layers=1, model__num_neurons=64;, score=0.991 total time=  42.2s
[CV 3/5] END batch_size=16, model__activation=relu, model__dropout_rate=0.1, model__learning_rate=0.01, model__num_layers=1, model__num_neurons=64;, score=0.991 total time=  43.5s
[CV 4/5] END batch_size=16, model__activation=relu, model__dropout_rate=0.1, model__learning_rate=0.01, model__num_layers=1, model__num_neurons=64;, score=0.994 total time=  44.1s
[CV 5/5] END batch_size=16, model__activation=relu, model__dropout_rate=0.1, model__learning_rate=0.01, model__num_layers=1, model__num_neurons=64;, score=0.986 total time=  42.8s
[CV 1/5] END batch_size=16, model__a

In [ ]:
print("dataframe.head: ")
print(X.head())
print(y.head())